In [1]:
import pandas as pd
from transformers import pipeline

In [2]:
df = pd.read_csv("data/bbc-news/bbc_news.csv")

df.head()

,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...


In [3]:
df["text"] = df["title"] + " " + df["description"]

In [4]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

Device set to use mps:0


In [5]:
topics = [
    "politics",
    "business",
    "technology",
    "sports",
    "entertainment",
    "world news"
]

In [6]:
text = df["text"].iloc[0]

result = classifier(text[:512], topics)

print(result["labels"][0])

politics


In [7]:
df = df.dropna()

df["text"] = df["title"] + ". " + df["description"]

df[["title","text"]].head()

,title,text
0,Ukraine: Angry Zelensky vows to punish Russian...,Ukraine: Angry Zelensky vows to punish Russian...
1,War in Ukraine: Taking cover in a town under a...,War in Ukraine: Taking cover in a town under a...
2,Ukraine war 'catastrophic for global food',Ukraine war 'catastrophic for global food'. On...
3,Manchester Arena bombing: Saffie Roussos's par...,Manchester Arena bombing: Saffie Roussos's par...
4,Ukraine conflict: Oil price soars to highest l...,Ukraine conflict: Oil price soars to highest l...


In [8]:
# from transformers import pipeline
# summarizer = pipeline("summarization")
# import transformers
# print(transformers.__version__)
from transformers import pipeline

In [9]:
from transformers import pipeline

summarizer = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

Device set to use mps:0


In [10]:
def summarize_news(article):

    prompt = f"summarize: {article}"

    result = summarizer(
        prompt,
        max_new_tokens=40
    )

    return result[0]["generated_text"]

In [11]:
article = df["text"].iloc[0]

print("Original Article:")
print(article)

summary = summarize_news(article)

print("\nSummary:")
print(summary)

Original Article:
Ukraine: Angry Zelensky vows to punish Russian atrocities. The Ukrainian president says the country will not forgive or forget those who murder its civilians.

Summary:
Ukraine's president vows to punish Russian atrocities


In [12]:
summaries = []

for article in df["text"].head(10):

    summaries.append(summarize_news(article))


df_sample = df.head(10).copy()

df_sample["summary"] = summaries

df_sample[["title","summary"]]

,title,summary
0,Ukraine: Angry Zelensky vows to punish Russian...,Ukraine's president vows to punish Russian atr...
1,War in Ukraine: Taking cover in a town under a...,"Jeremy Bowen was on the frontline in Irpin, Uk..."
2,Ukraine war 'catastrophic for global food',Ukraine war 'catastrophic for global food supply'
3,Manchester Arena bombing: Saffie Roussos's par...,"The parents of Saffie Roussos, the youngest vi..."
4,Ukraine conflict: Oil price soars to highest l...,Oil price soars to highest level since 2008
5,Ukraine war: PM to hold talks with world leade...,Boris Johnson is to hold talks with world lead...
6,Ukraine war: UK grants 50 Ukrainian refugee vi...,"Home Secretary Theresa May says she is ""surgin..."
7,TikTok limits services as Netflix pulls out of...,TikTok has announced that it will no longer of...
8,"Covid: Fourth jab for Scotland's vulnerable, a...",Covid: The fourth jab for Scotland's vulnerabl...
9,Protests across Russia see thousands detained,Protests across Russia have seen thousands det...


In [13]:
df_sample.to_csv("bbc_news_summaries.csv", index=False)

print("Saved file: bbc_news_summaries.csv")

Saved file: bbc_news_summaries.csv


In [14]:
from transformers import pipeline

topic_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

topics = [
    "politics",
    "business",
    "technology",
    "sports",
    "entertainment",
    "world news"
]

Device set to use mps:0


In [15]:
article = df["text"].iloc[0]

result = topic_classifier(article[:512], topics)

print("Topic:", result["labels"][0])

Topic: politics


In [16]:
from collections import Counter
import re

def extract_keywords(text):

    words = re.findall(r'\b[a-zA-Z]{5,}\b', text.lower())

    common = Counter(words).most_common(5)

    return [w[0] for w in common]

In [17]:
def analyze_news(article):

    topic = topic_classifier(article[:512], topics)["labels"][0]

    summary = summarize_news(article)

    keywords = extract_keywords(article)

    return topic, summary, keywords
    

In [18]:
article = df["text"].iloc[0]

topic, summary, keywords = analyze_news(article)

print("Topic:", topic)
print("Summary:", summary)
print("Keywords:", keywords)

Topic: politics
Summary: Ukraine's president vows to punish Russian atrocities
Keywords: ['ukraine', 'angry', 'zelensky', 'punish', 'russian']


In [19]:
print("AI NEWS ANALYZER")
print("-------------------")

print("Topic:", topic)
print("\nSummary:")
print(summary)

print("\nKeywords:", ", ".join(keywords))

AI NEWS ANALYZER
-------------------
Topic: politics

Summary:
Ukraine's president vows to punish Russian atrocities

Keywords: ukraine, angry, zelensky, punish, russian
